In [4]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

C:\Users\chitr\AppData\Local\Temp\ipykernel_18724\4099195310.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
c:\Users\chitr\Documents\Projects\ai-study-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
loader = DirectoryLoader(
    r"C:\Users\chitr\Documents\Projects\ai-study-assistant\data\ai_study_assistant_course_dataset",
    glob="**/*.txt",          
    loader_cls=TextLoader,    
    show_progress=True        
)

In [5]:
documents = loader.load()

100%|██████████| 9/9 [00:00<00:00, 129.05it/s]


In [6]:
len(documents)

9

In [22]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)


In [23]:
docs = text_splitter.split_documents(documents)

In [24]:
len(docs)

18

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5891.68it/s]


In [ ]:
# vector_store = Chroma.from_documents(
#     documents=docs,
#     embedding=embeddings,
#     persist_directory="./student_content.db"
# )

In [6]:
# loading code 

vector_store = Chroma(
    persist_directory="./student_content.db",
    embedding_function=embeddings
)

In [7]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 6, "lambda_mult": 0.25})

In [11]:
responses = retriever.invoke("What is ReActive Agent")

In [9]:
len(responses)

6

In [12]:
for r in responses:
    print(r.page_content)

    print()

    print("-"* 50)

ReAct: Reasoning and Acting

ReAct is a pattern for combining model reasoning with actions. The model decides what to do, calls a tool, observes the result, and can then decide what to do next.

A simplified loop is:

User
-> model decision
-> action
-> observation
-> model decision
-> final answer

For example, a user may ask:
"Explain RAG and remind me to revise it tomorrow."

The agent could first search the course material for RAG, observe the retrieved information, and then create a reminder. After the tool results are available, it produces a final response.

The important concept is that tool results can influence the next decision. This is different from a fixed pipeline where every request always follows exactly the same sequence.

--------------------------------------------------
The key idea is decision-making around actions. A tool-using agent needs clear tool definitions, controlled execution, and limits that prevent unnecessary or endless actions.

Not every request requ